# 🧪 Interactive Test Suite: Kaggle All-in-One AI Studio
Toàn bộ kết quả thử nghiệm (**Text, Audio, Image, Video**) sẽ được **hiển thị trực tiếp ngay dưới cell** thông qua `IPython.display` mà không sinh thêm bất kỳ file rác (artifact) nào trên ổ đĩa!
- 👁️ **VLM**: Hiển thị hội thoại định dạng Markdown
- 🔊 **TTS**: Audio Player tương tác trực tiếp
- 🎙️ **STT**: Kết quả nhận diện chữ từ âm thanh trong RAM
- 🖼️ **Image**: Hiển thị ảnh FLUX.1 trực tiếp
- 🎬 **Video**: Trình phát Video HTML5 trực tiếp

In [ ]:
# 1. Khởi tạo & Kiểm tra kết nối Endpoint
import os
import json
import base64
import time
import requests
from IPython.display import display, Markdown, Audio, Image, HTML

def get_base_url():
    for p in ["public_url.txt", "/kaggle/working/public_url.txt", "kaggle/all-in-one/public_url.txt"]:
        if os.path.exists(p):
            with open(p) as f:
                url = f.read().strip()
                if url.startswith("http"):
                    return url.rstrip("/")
    return "http://localhost:8000/v1"

BASE_URL = get_base_url()
ROOT_URL = BASE_URL.replace("/v1", "")
print(f"🎯 Target Base URL: {BASE_URL}")

# Kiểm tra trạng thái hệ thống
try:
    resp = requests.get(f"{ROOT_URL}/health", timeout=10)
    if resp.status_code == 200:
        health_data = resp.json()
        vram = health_data.get("vram_status", {})
        display(Markdown(f"### ✅ Hệ thống Online: `{health_data.get('service')}`"))
        gpu0_info = vram.get('gpu_0', {})
        gpu1_info = vram.get('gpu_1', {})
        display(Markdown(f"**GPU 0**: `{gpu0_info.get('allocated_gb', 0)}GB / {gpu0_info.get('total_gb', 0)}GB` | **GPU 1**: `{gpu1_info.get('allocated_gb', 0)}GB / {gpu1_info.get('total_gb', 0)}GB`"))
    else:
        print(f"⚠️ Health check trả về mã: {resp.status_code}")
except Exception as e:
    print(f"❌ Không thể kết nối tới server: {e}")

## 👁️ 1. Test Vision-Language Model (VLM / Chat Completions)
Gửi yêu cầu tới `/v1/chat/completions` và hiển thị phản hồi Markdown trực tiếp.

In [ ]:
# 2. Test VLM Chat
prompt = "Xin chào! Bạn là ai và có thể giúp gì cho tôi trong studio này?"

payload = {
    "model": "auto",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 200,
    "temperature": 0.7,
}

t0 = time.time()
resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=120)
elapsed = time.time() - t0

if resp.status_code == 200:
    data = resp.json()
    reply = data["choices"][0]["message"]["content"]
    display(Markdown(f"> **User:** *{prompt}*\n\n**Assistant ({elapsed:.2f}s):**\n\n{reply}"))
else:
    print(f"❌ Lỗi ({resp.status_code}): {resp.text}")

## 🔊 2. Test Text-to-Speech (TTS - Kokoro-82M Full FP16)
Sinh giọng nói đọc văn bản và **nhúng trực tiếp audio player** ngay bên dưới. Không lưu file vào đĩa!

In [ ]:
# 3. Test TTS
tts_text = "Xin chào! Hệ thống Studio đã sẵn sàng tạo ảnh, video và trò chuyện cùng bạn."

payload = {
    "model": "kokoro-82m",
    "input": tts_text,
    "voice": "af_heart",
    "response_format": "wav",
}

t0 = time.time()
resp = requests.post(f"{BASE_URL}/audio/speech", json=payload, timeout=60)
elapsed = time.time() - t0

if resp.status_code == 200:
    generated_audio_bytes = resp.content
    display(Markdown(f"**🔊 Audio đã tạo thành công trong {elapsed:.2f}s ({len(generated_audio_bytes):,} bytes):**"))
    display(Audio(data=generated_audio_bytes, autoplay=False))
else:
    print(f"❌ Lỗi ({resp.status_code}): {resp.text}")

## 🎙️ 3. Test Speech-to-Text (STT - Whisper-large-v3-turbo Full FP16)
Gửi thẳng mảng audio bytes vừa sinh ở trên qua bộ nhớ RAM vào `/v1/audio/transcriptions` để nhận diện chữ.

In [ ]:
# 4. Test STT
if 'generated_audio_bytes' in locals() and generated_audio_bytes:
    t0 = time.time()
    files = {"file": ("audio.wav", generated_audio_bytes, "audio/wav")}
    data = {"model": "whisper-large-v3-turbo"}
    resp = requests.post(f"{BASE_URL}/audio/transcriptions", files=files, data=data, timeout=60)
    elapsed = time.time() - t0

    if resp.status_code == 200:
        res_data = resp.json()
        display(Markdown(f"**🎙️ Nhận diện thành công trong {elapsed:.2f}s (Ngôn ngữ: `{res_data.get('language')}`):**"))
        display(Markdown(f"> *{res_data.get('text')}*"))
    else:
        print(f"❌ Lỗi ({resp.status_code}): {resp.text}")
else:
    print("⚠️ Hãy chạy cell TTS phía trên để có dữ liệu âm thanh kiểm thử.")

## 🖼️ 4. Test Image Generation (FLUX.1-schnell 4-bit)
Sinh ảnh chất lượng cao 1024x1024 và **hiển thị trực tiếp ảnh trong cell** mà không lưu file PNG ra đĩa.

In [ ]:
# 5. Test Image Generation
image_prompt = "A majestic mechanical tiger with glowing neon circuitry, cyberpunk Tokyo rooftop, cinematic lighting, 8k render"

payload = {
    "model": "flux-1-schnell",
    "prompt": image_prompt,
    "size": "1024x1024",
    "steps": 4,
    "response_format": "b64_json",
}

display(Markdown(f"🎨 Đang yêu cầu FLUX.1 sinh ảnh: *'{image_prompt}'*..."))
t0 = time.time()
resp = requests.post(f"{BASE_URL}/images/generations", json=payload, timeout=300)
elapsed = time.time() - t0

if resp.status_code == 200:
    img_data = resp.json()
    b64_img = img_data["data"][0].get("b64_json")
    if b64_img:
        display(Markdown(f"**✅ Sinh ảnh thành công trong {elapsed:.2f}s:**"))
        display(Image(data=base64.b64decode(b64_img)))
    else:
        print("⚠️ Không có chuỗi base64:", img_data)
else:
    print(f"❌ Lỗi ({resp.status_code}): {resp.text}")

## 🎬 5. Test Video Generation (Wan2.1-14B / WanPipeline)
Sinh video điện ảnh và **hiển thị trình phát video HTML5 trực tiếp dưới cell**. Không lưu file MP4 ra đĩa!

In [ ]:
# 6. Test Video Generation
video_prompt = "A futuristic cyberpunk car driving on a rainy neon highway, cinematic lighting, 4k"

payload = {
    "model": "wan-2.1-14b",
    "prompt": video_prompt,
    "num_frames": 25,
    "width": 768,
    "height": 512,
}

display(Markdown(f"🎬 Đang yêu cầu Wan2.1 sinh video: *'{video_prompt}'*..."))
t0 = time.time()
resp = requests.post(f"{BASE_URL}/videos/generations", json=payload, timeout=600)
elapsed = time.time() - t0

if resp.status_code == 200:
    vid_data = resp.json()
    b64_vid = vid_data["data"][0].get("b64_json")
    if b64_vid:
        display(Markdown(f"**✅ Sinh video thành công trong {elapsed:.2f}s:**"))
        video_html = f'''
        <video width="768" height="512" controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
            <source src="data:video/mp4;base64,{b64_vid}" type="video/mp4">
            Trình duyệt của bạn không hỗ trợ thẻ video HTML5.
        </video>
        '''
        display(HTML(video_html))
    else:
        print("⚠️ Không có chuỗi base64:", vid_data)
else:
    print(f"❌ Lỗi ({resp.status_code}): {resp.text}")